# Task 2.1 - Neural Network in PyTorch
**Skill Set Go EduTech - AI/ML Internship - Week 2**

**Goal:** build a small feed-forward neural network in PyTorch and understand every step of training:
what goes in (features), what comes out (prediction), how error is measured (loss), and how the weights are updated.

**Task chosen:** binary classification on the Breast Cancer Wisconsin dataset (30 numeric features per patient sample,
label = malignant or benign). It is small and purely numerical, so I can focus on the mechanics of the network
instead of on data cleaning.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pip install torch torchvision scikit-learn matplotlib pandas

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# fixed seeds so my results are repeatable
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
os.makedirs("results", exist_ok=True)

from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# fixed seeds so my results are repeatable
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
os.makedirs("results", exist_ok=True)

SyntaxError: invalid syntax (3750164783.py, line 6)

## 1. Dataset setup
Load the data and look at it before doing anything else: shape, classes and class balance.

In [ ]:
data = load_breast_cancer()
X = data.data            # features, shape (569, 30)
y = data.target          # labels, 0 = malignant, 1 = benign

print("Features shape:", X.shape)
print("Labels shape  :", y.shape)
print("Class names   :", list(data.target_names))
print(pd.Series(y).map({0: "malignant", 1: "benign"}).value_counts())
pd.DataFrame(X, columns=data.feature_names).iloc[:, :6].describe().round(2)

The classes are not perfectly balanced (about 63% benign) but not extreme either, so plain accuracy is a fair first metric.

### Train / validation split and scaling
Features have very different ranges (e.g. area is in the hundreds, smoothness below 1), which makes training harder.
I scale them with `StandardScaler`. **The scaler is fitted on the training data only** and then applied to the
validation data, otherwise information from the validation set would leak into training.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # learn mean/std from training data only
X_val = scaler.transform(X_val)           # reuse the same mean/std

# convert to tensors: float32 for features, float32 with shape (N, 1) for the targets
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)

print("X_train:", X_train_t.shape, X_train_t.dtype)
print("y_train:", y_train_t.shape, y_train_t.dtype)
print("X_val  :", X_val_t.shape)
print("y_val  :", y_val_t.shape)

## 2. Dataset and DataLoader
A `Dataset` tells PyTorch how to get one sample; a `DataLoader` groups samples into shuffled mini-batches.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

train_loader = DataLoader(TabularDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
val_loader   = DataLoader(TabularDataset(X_val_t, y_val_t), batch_size=32, shuffle=False)

xb, yb = next(iter(train_loader))
print("One batch of features:", xb.shape)   # (32, 30)
print("One batch of targets :", yb.shape)   # (32, 1)

## 3. Model definition
A small feed-forward network: 30 inputs -> 16 hidden -> 8 hidden -> 1 output.
ReLU is used between the layers so the network can learn non-linear patterns.
The last layer has **no** activation: it outputs a raw score (a *logit*), and the sigmoid is applied inside the loss
function during training (and by me when I want a probability).

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.net(x)

model = SimpleNet(in_features=X_train.shape[1]).to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Loss function and optimizer
* **Loss:** `BCEWithLogitsLoss` - the standard choice for binary classification. It combines a sigmoid and binary
  cross-entropy in one numerically stable step, so the model can output raw logits.
* **Optimizer:** `Adam` with learning rate 0.001 - a common default that adapts the step size per weight, so it usually
  works without much tuning on a small problem like this.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 5. Training loop
Every batch goes through the same 5 steps:
1. clear old gradients (`zero_grad`)
2. forward pass (inputs -> predictions)
3. compute the loss (predictions vs. real targets)
4. backpropagate (`loss.backward()` fills each weight's gradient)
5. update the weights (`optimizer.step()`)

After each epoch the model is switched to `eval()` mode and tested on the validation set **without** updating any
weights (`torch.no_grad()`). Train and validation loss are both recorded.

In [ ]:
epochs = 50
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    # ---------- training ----------
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()            # 1. clear gradients from the previous batch
        logits = model(xb)               # 2. forward pass
        loss = criterion(logits, yb)     # 3. loss
        loss.backward()                  # 4. gradients of the loss w.r.t. every weight
        optimizer.step()                 # 5. update the weights

        running_loss += loss.item() * xb.size(0)
    train_loss = running_loss / len(train_loader.dataset)

    # ---------- validation (no weight updates) ----------
    model.eval()
    val_running, correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            val_running += criterion(logits, yb).item() * xb.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == yb).sum().item()
    val_loss = val_running / len(val_loader.dataset)
    val_acc = correct / len(val_loader.dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{epochs} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val acc {val_acc:.4f}")

## 6. Training and validation loss

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["train_loss"], label="Training loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Task 2.1 - Loss per epoch")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/loss_curve.png", dpi=150)
plt.show()

## 7. Evaluation and sample predictions
Compare the model's predictions with the real targets on samples it did not train on.

In [ ]:
model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X_val_t.to(device))).cpu().squeeze(1)
    all_preds = (probs > 0.5).int()

val_acc = (all_preds.numpy() == y_val).mean()
print(f"Final validation accuracy: {val_acc:.4f}")

sample = pd.DataFrame({
    "actual": y_val[:10],
    "predicted_prob_benign": probs[:10].numpy().round(3),
    "predicted": all_preds[:10].numpy(),
})
sample["correct"] = sample["actual"] == sample["predicted"]
sample   # 1 = benign, 0 = malignant

In [ ]:
results = {
    "task": "2.1",
    "dataset": "Breast Cancer Wisconsin (sklearn), 569 samples, 30 features",
    "model": "30-16-8-1 feed-forward, ReLU",
    "loss": "BCEWithLogitsLoss",
    "optimizer": "Adam (lr=0.001)",
    "batch_size": 32,
    "epochs": epochs,
    "final_train_loss": round(history["train_loss"][-1], 4),
    "final_val_loss": round(history["val_loss"][-1], 4),
    "val_accuracy": round(float(val_acc), 4),
}
with open("results/results.json", "w") as f:
    json.dump(results, f, indent=2)
results

## 8. My interpretation of the results
*(Write this in your own words after you have seen your own output. Answer briefly:)*

* What were the final train loss, validation loss and validation accuracy?
* Do train and validation loss move together, or does validation loss start rising while train loss keeps falling?
* How many predictions out of the 10 samples were right, and does that match the overall accuracy?
* What would I change next (more epochs, different learning rate, more hidden units)?

**My notes:**

-

## 9. Can I explain it line by line?
Before submitting, check that you can explain:
- what `xb` and `yb` are and their shapes
- what `model(xb)` returns (logits, not probabilities)
- why `zero_grad()` comes first
- what `loss.backward()` actually computes
- what `optimizer.step()` changes
- why `model.eval()` and `torch.no_grad()` are used for validation